---
## ⚙️ 实验配置

**修改下方参数后运行此单元格**

In [2]:
# ═══════════════════════════════════════════════════════════════
# 实验配置 - 修改此处参数
# ═══════════════════════════════════════════════════════════════

CONFIG = {
    # --- 攻击参数 ---
    "poison_rate": 0.2,           # 恶意客户端毒化比例 (0~1)
    "scaling_factor": 4.0,         # 模型替换放大因子
    "backdoor_boost_weight": 0.1,  # 后门增强损失权重
    "epsilon": 0.1,                # 触发器注入强度
    "target_label": 0,             # 后门目标类
    "freq_strategy": "ANB",        # 攻击策略: "ANB" 或 "FIXED"
    
    # --- 训练参数 ---
    "num_rounds": 10,              # 联邦学习轮数
    "local_epochs": 3,             # 本地训练轮数
    "learning_rate": 0.01,         # 学习率
    "batch_size": 32,              # 批次大小
    "seed": 42,                    # 随机种子
    
    # --- 防御参数 ---
    "defense_enabled": True,       # 是否启用防御
    "defense_method": "hdbscan",   # 防御方法: hdbscan, kmeans, dbscan
    
    # --- 其他 ---
    "num_clients": 10,             # 客户端数量
    
    # --- 消融实验开关 ---
    "use_phased_chaos": True,      # 动态相位调度
    "use_spectral_smoothing": True, # 高斯扩散
    "use_freq_sharding": True,     # 频率分片
    "use_dual_routing": True,      # 双域路由
}

# 打印配置
print("="*60)
print("实验配置")
print("="*60)
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
print("="*60)

实验配置
  poison_rate: 0.2
  scaling_factor: 4.0
  backdoor_boost_weight: 0.1
  epsilon: 0.1
  target_label: 0
  freq_strategy: ANB
  num_rounds: 10
  local_epochs: 3
  learning_rate: 0.01
  batch_size: 32
  seed: 42
  defense_enabled: True
  defense_method: hdbscan
  num_clients: 10
  use_phased_chaos: True
  use_spectral_smoothing: True
  use_freq_sharding: True
  use_dual_routing: True


---
## 🔧 环境设置

In [3]:
# 设置项目路径
import os
import sys
from pathlib import Path

# 项目根目录（colab 的父目录）
NOTEBOOK_PATH = Path(__file__).absolute() if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_PATH.parent.parent if 'colab' in str(NOTEBOOK_PATH) else Path('D:/1研/毕业论文/SAFB')

# 切换到项目根目录
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print(f"项目根目录: {PROJECT_ROOT}")
print(f"当前工作目录: {os.getcwd()}")

FileNotFoundError: [Errno 2] No such file or directory: 'D:/1研/毕业论文/SAFB'

In [ ]:
# 检查 GPU
import torch

print("="*60)
print("环境信息")
print("="*60)
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 设备: {torch.cuda.get_device_name(0)}")
    print(f"GPU 内存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ 警告: 未检测到 GPU，实验将运行缓慢")
print("="*60)

In [ ]:
# 验证模块导入
print("验证模块导入...")

try:
    from config import Config, load_config
    print("  ✓ config")
    
    from core.attacks import AdaptiveNebulaBackdoor, FrequencyBackdoor
    print("  ✓ core.attacks")
    
    from core.defenses import aggregate_with_defense
    print("  ✓ core.defenses")
    
    from federated.client import Client, create_clients
    print("  ✓ federated.client")
    
    from federated.server import Server, federated_training
    print("  ✓ federated.server")
    
    print("\n✓ 所有模块导入成功!")
    
except ImportError as e:
    print(f"\n✗ 导入失败: {e}")

---
## 🚀 运行实验

In [ ]:
# 构建命令并运行
cmd = f"""python main.py \
    --poison-rate {CONFIG['poison_rate']} \
    --scaling-factor {CONFIG['scaling_factor']} \
    --backdoor-boost-weight {CONFIG['backdoor_boost_weight']} \
    --epsilon {CONFIG['epsilon']} \
    --target-label {CONFIG['target_label']} \
    --freq-strategy {CONFIG['freq_strategy']} \
    --num-rounds {CONFIG['num_rounds']} \
    --local-epochs {CONFIG['local_epochs']} \
    --learning-rate {CONFIG['learning_rate']} \
    --batch-size {CONFIG['batch_size']} \
    --seed {CONFIG['seed']} \
    --num-clients {CONFIG['num_clients']} \
    --defense-enabled {1 if CONFIG['defense_enabled'] else 0} \
    --defense-method {CONFIG['defense_method']} \
    --use-phased-chaos {1 if CONFIG['use_phased_chaos'] else 0} \
    --use-spectral-smoothing {1 if CONFIG['use_spectral_smoothing'] else 0} \
    --use-freq-sharding {1 if CONFIG['use_freq_sharding'] else 0} \
    --use-dual-routing {1 if CONFIG['use_dual_routing'] else 0}"""

print("执行命令:")
print(cmd)
print("\n" + "="*60)
print("开始训练...")
print("="*60 + "\n")

# 运行实验
get_ipython().run_cell_magic('bash', '', cmd)

---
## 📊 查看结果

In [ ]:
import json
import os
from pathlib import Path

# 查找结果文件
results_dir = Path('results')
result_files = list(results_dir.glob('history_*.json')) if results_dir.exists() else []

if result_files:
    # 使用最新的结果文件
    result_file = sorted(result_files, key=lambda x: x.stat().st_mtime)[-1]
    
    with open(result_file, 'r') as f:
        history = json.load(f)
    
    print("="*60)
    print("实验结果")
    print("="*60)
    print(f"结果文件: {result_file.name}")
    print(f"训练轮数: {len(history['test_acc'])}")
    print("-"*60)
    print(f"最终 ACC (Clean Accuracy):  {history['test_acc'][-1]:.2%}")
    print(f"最终 ASR (Single Trigger):  {history['test_asr'][-1]:.2%}")
    
    if history.get('test_asr_multi'):
        print(f"最终 ASR (Multi-Trigger):   {history['test_asr_multi'][-1]:.2%}")
    
    if history.get('defense_bypass_rate'):
        print(f"最终 Bypass Rate:           {history['defense_bypass_rate'][-1]:.2%}")
    
    print("-"*60)
    
    # 判断是否达标
    acc_ok = history['test_acc'][-1] >= 0.60
    asr_ok = history['test_asr'][-1] >= 0.85
    
    print("达标情况:")
    print(f"  ACC ≥ 60%:     {'✓ 达标' if acc_ok else '✗ 未达标'} ({history['test_acc'][-1]:.2%})")
    print(f"  ASR ≥ 85%:     {'✓ 达标' if asr_ok else '✗ 未达标'} ({history['test_asr'][-1]:.2%})")
    
    print("="*60)
else:
    print("⚠️ 未找到结果文件")
    print(f"结果目录: {results_dir.absolute()}")

In [ ]:
# 可视化结果
import matplotlib.pyplot as plt
import numpy as np

if 'history' in dir():
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    rounds = range(1, len(history['test_acc']) + 1)
    
    # ACC 曲线
    axes[0, 0].plot(rounds, [x*100 for x in history['test_acc']], 'b-', linewidth=2, marker='o', markersize=4)
    axes[0, 0].axhline(y=60, color='g', linestyle='--', label='Target (60%)')
    axes[0, 0].set_xlabel('Round')
    axes[0, 0].set_ylabel('Accuracy (%)')
    axes[0, 0].set_title('Clean Accuracy')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].set_ylim([0, 100])
    
    # ASR 曲线
    axes[0, 1].plot(rounds, [x*100 for x in history['test_asr']], 'r-', linewidth=2, marker='o', markersize=4, label='Single')
    if history.get('test_asr_multi'):
        axes[0, 1].plot(rounds, [x*100 for x in history['test_asr_multi']], 'r--', linewidth=2, marker='s', markersize=4, label='Multi')
    axes[0, 1].axhline(y=85, color='g', linestyle='--', label='Target (85%)')
    axes[0, 1].set_xlabel('Round')
    axes[0, 1].set_ylabel('ASR (%)')
    axes[0, 1].set_title('Attack Success Rate')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_ylim([0, 105])
    
    # Bypass Rate 曲线
    if history.get('defense_bypass_rate'):
        axes[1, 0].plot(rounds, [x*100 for x in history['defense_bypass_rate']], 'g-', linewidth=2, marker='o', markersize=4)
        axes[1, 0].axhline(y=70, color='orange', linestyle='--', label='Target (70%)')
        axes[1, 0].set_xlabel('Round')
        axes[1, 0].set_ylabel('Bypass Rate (%)')
        axes[1, 0].set_title('Defense Bypass Rate')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        axes[1, 0].set_ylim([0, 105])
    
    # Training Loss 曲线
    if history.get('train_loss'):
        axes[1, 1].plot(rounds, history['train_loss'], 'purple', linewidth=2, marker='o', markersize=4)
        axes[1, 1].set_xlabel('Round')
        axes[1, 1].set_ylabel('Loss')
        axes[1, 1].set_title('Training Loss')
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('results/experiment_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n✓ 图表已保存到 results/experiment_results.png")
else:
    print("⚠️ 请先运行实验")

---
## 🔄 批量实验（可选）

In [ ]:
# 批量实验配置
BATCH_CONFIGS = [
    {"name": "solution_v1", "poison_rate": 0.75, "scaling_factor": 5.5, "boost": 0.3},
    {"name": "solution_v2", "poison_rate": 0.80, "scaling_factor": 5.0, "boost": 0.3},
    {"name": "solution_v3", "poison_rate": 0.70, "scaling_factor": 6.0, "boost": 0.3},
]

RUN_BATCH = False  # 改为 True 以运行批量实验

if RUN_BATCH:
    import subprocess
    batch_results = []
    
    for i, cfg in enumerate(BATCH_CONFIGS):
        print(f"\n{'='*60}")
        print(f"[{i+1}/{len(BATCH_CONFIGS)}] Running: {cfg['name']}")
        print('='*60)
        
        args = [
            sys.executable, "main.py",
            "--poison-rate", str(cfg['poison_rate']),
            "--scaling-factor", str(cfg['scaling_factor']),
            "--backdoor-boost-weight", str(cfg['boost']),
            "--num-rounds", str(CONFIG['num_rounds']),
            "--local-epochs", str(CONFIG['local_epochs']),
            "--seed", str(CONFIG['seed']),
        ]
        
        subprocess.run(args)
        
        # 收集结果
        result_files = list(results_dir.glob('history_*.json'))
        if result_files:
            latest = sorted(result_files, key=lambda x: x.stat().st_mtime)[-1]
            with open(latest, 'r') as f:
                h = json.load(f)
            batch_results.append({
                "name": cfg['name'],
                "acc": h['test_acc'][-1],
                "asr": h['test_asr'][-1],
            })
    
    # 打印汇总
    print("\n" + "="*60)
    print("批量实验结果汇总")
    print("="*60)
    print(f"{'Name':<15} {'ACC':>10} {'ASR':>10}")
    print("-"*35)
    for r in batch_results:
        print(f"{r['name']:<15} {r['acc']:>10.2%} {r['asr']:>10.2%}")
    print("="*60)
else:
    print("批量实验未启用。设置 RUN_BATCH = True 以运行。")

---
## 📋 参数说明

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `poison_rate` | 0.75 | 恶意客户端毒化比例 |
| `scaling_factor` | 5.5 | 模型替换放大因子 |
| `backdoor_boost_weight` | 0.3 | 后门增强损失权重 |
| `num_rounds` | 30 | 联邦学习轮数 |
| `local_epochs` | 3 | 本地训练轮数 |
| `freq_strategy` | ANB | 攻击策略 |

**预期结果**：ACC ≥ 60%, ASR ≥ 85%